# Basic MLP

Binary classification of the two moons dataset with a small MLP. Set `engine_str` to `'raxis'` or `'torch'`: the rest of the notebook is the same code.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Make `examples.utils` importable from this directory
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from examples.utils.data import split_dataset, DataLoader, DataLoaderCallback
from examples.utils.train import Trainer, Evaluator

In [ ]:
supported_engines = ['raxis', 'torch']

In [ ]:
seed = 37
engine_str = 'raxis'
device = 'cpu'

In [ ]:
epochs = 50
batch_size = 4
lr = 0.1

In [ ]:
if engine_str == 'raxis':
    import raxis as engine
    from raxis import nn, optim
elif engine_str == 'torch':
    import torch as engine
    from torch import nn, optim
else:
    raise ValueError(f"engine_str must be one of {supported_engines}")

engine.manual_seed(seed)
np.random.seed(seed)
print("Engine:", engine_str)

In [ ]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=150, noise=0.1, random_state=seed)
X = X.astype(np.float32)
y = y.astype(np.float32)  # 0 or 1

plt.figure(figsize=(5, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, s=20, cmap='jet')
plt.show()

In [ ]:
class MultiLayerPerceptron(nn.Module):

    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            if module.weight.shape[0] == 1:  # last layer
                nn.init.xavier_uniform_(module.weight)
                with engine.no_grad():
                    module.weight *= 0.1  # make the logits layer less confident
            else:
                nn.init.kaiming_normal_(module.weight, nonlinearity='relu')
            nn.init.zeros_(module.bias)

    def forward(self, x: engine.Tensor):
        return self.mlp(x)

model = MultiLayerPerceptron()
model.to(device)
print("MLP Model:", model)
print("MLP Trainable Parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
class Transform(DataLoaderCallback):

    def __call__(self, data_loader: 'DataLoader', X_batch: np.ndarray, y_batch: np.ndarray):
        x = engine.tensor(X_batch, device=device)
        y = engine.tensor(y_batch, device=device)
        return x, y

train_loader = DataLoader(X, y, batch_size, transform=Transform())
print("Train batches:", len(train_loader))

In [ ]:
loss_fn = nn.MSELoss()  # nn.BCEWithLogitsLoss() could be used as well
optimizer = optim.SGD(model.parameters(), lr=lr)

trainer = Trainer(model, engine)
evaluator = Evaluator(accuracy=True, mode=Evaluator.BINARY)
trainer.compile(loss_fn, optimizer, evaluator)
history = trainer.fit(train_loader, epochs=epochs)

In [ ]:
trainer.plot(['accuracy', 'loss'])

In [ ]:
# Decision boundary
model.eval()
h = 0.25
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Xmesh = np.c_[xx.ravel(), yy.ravel()].astype(np.float32)

inputs = engine.tensor(Xmesh, device=device)
with engine.no_grad():
    scores = model(inputs)
Z = (scores.cpu().numpy() > 0.5).reshape(xx.shape)

plt.figure()
plt.contourf(xx, yy, Z, cmap=plt.cm.Spectral, alpha=0.8)
plt.scatter(X[:, 0], X[:, 1], c=y, s=40, cmap=plt.cm.Spectral)
plt.xlim(xx.min(), xx.max())
plt.ylim(yy.min(), yy.max())
plt.show()